In [2]:
from langchain_tavily import TavilySearch

tool = TavilySearch(max_results=2)
tools = [tool]
tool.invoke("What's a node 'node' in LangGraph")

{'query': "What's a node 'node' in LangGraph",
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.ibm.com/think/topics/langgraph',
   'title': 'What is LangGraph? | IBM',
   'content': 'Nodes: In LangGraph, nodes represent individual components or agents within an AI workflow. Nodes can be thought of as “actors” that interact with each other in a specific way. For example, to add nodes for tool calling, one can use the ToolNode. Another example, the next node, refers to the node that will be executed following the current one. [...] Workflows: Workflows are the sequences of node interactions that define an AI workflow. By arranging nodes into a workflow, users can create more complex and dynamic workflows that use the strengths of individual components.\n\nAPIs: LangGraph provides a set of APIs that enable users to interact with its components in a programmatic way. Users can use an API key, add new nodes, modify existing workflows and retri

In [3]:
import os

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage

api_key = os.getenv("QWEN_API_KEY")
if not api_key:
    raise ValueError("Get QWEN_API_KEY Failed!")

llm = ChatOpenAI(
    model="qwen3.7-flash",
    api_key=api_key,
    base_url="https://maas.qianwenaiapi.com/compatible-mode/v1",
    use_responses_api=False,
    extra_body={"enable_thinking":False}
)


In [4]:
from typing import Annotated

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list,add_messages]

graph_builder=StateGraph(State)

llm_with_tools = llm.bind_tools(tools=tools)

def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node("chatbot",chatbot)

In [5]:
import json

from langchain_core.messages import ToolMessage

class BasicToolNode:

    def __init__(self,tools: list) -> None:
        self.tools_by_name={tool.name:tool for tool in tools}

    def __call__(self, inputs: dict):
        if messages := inputs.get("messages",[]):
            message=messages[-1]
        else:
            raise ValueError("No message found in input!")
        outputs=[]
        for tool_call in message.tool_calls:
            tool_result=self.tools_by_name[tool_call["name"]].invoke(
                tool_call["args"]
            )
            outputs.append(
                ToolMessage(
                    content=json.dumps(tool_result),
                    name=tool_call["name"],
                    tool_call_id=tool_call["id"]
                )
            )
        return {"message":outputs}

tool_node=BasicToolNode(tools=[tool])
graph_builder.add_node("tools",tool_node)

In [6]:
def route_tools(
        state: State,
):
    if isinstance(state,list):
        ai_message=state[-1]
    elif messages:=state.get("messages",[]):
        ai_message=messages[-1]
    else:
        raise ValueError(f"No messages found in input state to tool_edge:{state}")
    if hasattr(ai_message,"tool_calls") and len(ai_message.tool_calls) >0:
        return "tools"
    return END

graph_builder.add_conditional_edges(
    "chatbot",
    route_tools,
    {"tools":"tools",END:END}
)

graph_builder.add_edge(START,"chatbot")
graph_builder.add_edge("tools","chatbot")
graph_builder.add_edge("chatbot",END)

graph=graph_builder.compile()

In [ ]:
def stream_graph_updates(user_input: str):
    for event in graph.stream({"messages":[{"role":"user","content":user_input}]}):#graph.stream()可以看到过程中节点输出结果
        print(event)


while True:
    try:
        user_input = input("User:")
        if user_input.lower() in ["quit","q","exit"]:
            print("Goodbye!")
            break

        stream_graph_updates(user_input)
    except:
        user_input="Who is Yanami Anna?"
        print("user:"+user_input)
        stream_graph_updates(user_input)
        break

Yanami: 
user:Who is Yanami Anna?
Yanami: 


TypeError: 'NoneType' object is not subscriptable